# ABSA Training With Google Drive Outputs

This notebook keeps source code in the Colab VM but saves model checkpoints, predictions, and metrics to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/ngocvo2511/NLP.git"
PROJECT_DIR = "/content/NLP"
OUTPUT_ROOT = "/content/drive/MyDrive/NLP_outputs"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["OUTPUT_ROOT"] = OUTPUT_ROOT

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!ls
print("Outputs will be saved to:", OUTPUT_ROOT)

In [ ]:
!pip install -q transformers accelerate datasets seqeval scikit-learn

In [ ]:
!python -m src.absa.validate_data --data-dir data

## Baseline Main Model: XLM-R BIO Raw

This is the main clean baseline. It uses raw model spans without heuristic post-processing.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-base \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-bio" \
  --tag-scheme bio \
  --class-weight none \
  --epochs 5 \
  --batch-size 8 \
  --max-length 256

!python -m src.absa.predict_transformer \
  --model-dir "$OUTPUT_ROOT/xlmr-bio" \
  --input data/dev.jsonl \
  --output "$OUTPUT_ROOT/xlmr_bio_dev_predictions.jsonl"

!python -m src.absa.evaluate \
  --gold data/dev.jsonl \
  --pred "$OUTPUT_ROOT/xlmr_bio_dev_predictions.jsonl" \
  --json-output "$OUTPUT_ROOT/xlmr_bio_dev_metrics.json"

## Clean Experiment: XLM-R BILOU Raw

BILOU is a clean boundary-modeling variant. Keep it only if it beats the BIO baseline.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-base \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-bilou" \
  --tag-scheme bilou \
  --class-weight none \
  --epochs 5 \
  --batch-size 8 \
  --max-length 256

!python -m src.absa.predict_transformer \
  --model-dir "$OUTPUT_ROOT/xlmr-bilou" \
  --input data/dev.jsonl \
  --output "$OUTPUT_ROOT/xlmr_bilou_dev_predictions.jsonl"

!python -m src.absa.evaluate \
  --gold data/dev.jsonl \
  --pred "$OUTPUT_ROOT/xlmr_bilou_dev_predictions.jsonl" \
  --json-output "$OUTPUT_ROOT/xlmr_bilou_dev_metrics.json"

## Clean Experiment: XLM-R BILOU With Train-Set Class Weights

This may increase recall for rare labels, but keep it only if exact F1 improves.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-base \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-bilou-weighted" \
  --tag-scheme bilou \
  --class-weight sqrt-balanced \
  --epochs 5 \
  --batch-size 8 \
  --max-length 256

!python -m src.absa.predict_transformer \
  --model-dir "$OUTPUT_ROOT/xlmr-bilou-weighted" \
  --input data/dev.jsonl \
  --output "$OUTPUT_ROOT/xlmr_bilou_weighted_dev_predictions.jsonl"

!python -m src.absa.evaluate \
  --gold data/dev.jsonl \
  --pred "$OUTPUT_ROOT/xlmr_bilou_weighted_dev_predictions.jsonl" \
  --json-output "$OUTPUT_ROOT/xlmr_bilou_weighted_dev_metrics.json"

All important artifacts are now in Google Drive under `MyDrive/NLP_outputs`.